# nb_transform_silver
Generic, metadata-driven bronze -> silver. Pipeline passes `entity` (and optionally `batch_id`).
Reads `meta_entities` (bronze table, business key) and `meta_columns` (select, rename, cast, transform).
Attach formula1 as the default lakehouse.

In [3]:
# PARAMETERS CELL (toggle "parameter cell" on this cell in Fabric so the pipeline can override)
entity = "sprints"
batch_id = ""

StatementMeta(, 211e6068-715d-4d31-9b22-cc64393ee9dc, 5, Finished, Available, Finished, False)

In [4]:
import re, uuid
from datetime import datetime
from functools import reduce
from pyspark.sql import functions as F

batch_id = batch_id or str(uuid.uuid4())
start_ts = datetime.utcnow()

TRANSFORMS = {"initcap": F.initcap, "upper": F.upper, "lower": F.lower, "trim": F.trim}
IDENT = re.compile(r"\w+")

# ---------- 1. read config ----------
cfg = (spark.table("meta_entities")
            .filter((F.col("entity") == entity) & (F.col("is_active") == 1))
            .first())
if cfg is None:
    raise ValueError(f"Entity '{entity}' not found or inactive in meta_entities")

cols = (spark.table("meta_columns")
             .filter((F.col("entity") == entity) & (F.col("is_active") == 1))
             .orderBy("ordinal").collect())
if not cols:
    raise ValueError(f"No active rows in meta_columns for '{entity}'")

keys = [k.strip() for k in (cfg.primary_key or "").split(",") if k.strip()]
bronze_table = cfg.bronze_table
silver_table = f"silver.{entity}"

status, rows_written, error_msg = "SUCCESS", None, None

def build_col(r):
    c = F.col(f"`{r.source_expr}`") if IDENT.fullmatch(r.source_expr) else F.expr(r.source_expr)
    c = c.cast(r.data_type)
    if r.transform:
        c = TRANSFORMS[r.transform](c)
    return c.alias(r.target_col)

try:
    df = spark.table(bronze_table)

    # ---------- 2. schema validation (fail fast on drift) ----------
    bronze_cols = set(df.columns)
    missing = [r.source_expr for r in cols if IDENT.fullmatch(r.source_expr) and r.source_expr not in bronze_cols]
    if missing:
        raise Exception(f"Schema drift in {entity}: bronze is missing {missing}")

    targets = {r.target_col for r in cols}
    bad_keys = [k for k in keys if k not in targets]
    if bad_keys:
        raise Exception(f"Primary key columns {bad_keys} are not in meta_columns for {entity}")

    # ---------- 3. select / rename / cast / transform ----------
    selected = df.select(*[build_col(r) for r in cols])

    # ---------- 4. business-key validation + dedupe ----------
    valid = selected
    if keys:
        valid = selected.filter(reduce(lambda a, b: a & b, [F.col(k).isNotNull() for k in keys]))
    final = valid.dropDuplicates(keys) if keys else valid.dropDuplicates()

    rows_in    = df.count()
    rows_valid = valid.count()

    # ---------- 5. write silver ----------
    (final.write.format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(silver_table))

    rows_written = spark.table(silver_table).count()
    print(f"[{entity}] bronze={rows_in}  null-key rejected={rows_in - rows_valid}  "
          f"duplicates removed={rows_valid - rows_written}  silver={rows_written}")

except Exception as e:
    status, error_msg = "FAILED", str(e)[:4000]
    raise

finally:
    log = spark.createDataFrame(
        [(batch_id, entity, "silver", status, rows_written, start_ts, datetime.utcnow(), error_msg)],
        "batch_id string, entity string, layer string, status string, rows_written long, "
        "start_ts timestamp, end_ts timestamp, error_msg string")
    log.write.format("delta").mode("append").saveAsTable("meta_run_log")

StatementMeta(, 211e6068-715d-4d31-9b22-cc64393ee9dc, 6, Finished, Available, Finished, False)

/tmp/ipykernel_6268/3168819476.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow()
/tmp/ipykernel_6268/3168819476.py:80: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(batch_id, entity, "silver", status, rows_written, start_ts, datetime.utcnow(), error_msg)],


In [6]:
display(spark.table("silver.sprints"))

StatementMeta(, 211e6068-715d-4d31-9b22-cc64393ee9dc, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3a8e2277-2aab-40bd-a24b-f52a65efe00c)

StatementMeta(, 211e6068-715d-4d31-9b22-cc64393ee9dc, 9, Finished, Available, Finished, False)